In [1]:
import pandas as pd

# Load Dataset 
df = pd.read_csv("email-spam-classification-dataset.csv")

# Prints the first 5 rows
print(df.head())

# Shows number of rows and number of blank rows
print(df.info())

   label                                               text
0      1  ounce feather bowl hummingbird opec moment ala...
1      1  wulvob get your medircations online qnb ikud v...
2      0   computer connection from cnn com wednesday es...
3      1  university degree obtain a prosperous future m...
4      0  thanks for all your answers guys i know i shou...
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 83448 entries, 0 to 83447
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   label   83448 non-null  int64 
 1   text    83448 non-null  object
dtypes: int64(1), object(1)
memory usage: 1.3+ MB
None


In [ ]:
# Check for rows where the text is just whitespace or empty
empty_text_rows = df[df['text'].str.strip() == ""]

print(f"Number of perfectly blank emails: {len(empty_text_rows)}")

# If it finds any, this line removes them:
# df = df[df['text'].str.strip() != ""]

Number of perfectly blank emails: 0


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

# 1. Separate the inputs (text) from the answers (labels)
X = df['text']
y = df['label']

# 2. Split into 80% training and 20% testing
X_train_text, X_test_text, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42) # random_state=42 ensures we get the exact same split every time we run it

print("Data successfully split!")

# 3. Initialize the Vectorizer
# 'stop_words=english' automatically removes useless words like 'a', 'an', 'the'
# 'max_features=5000' keeps only the top 5000 most important words to save memory
vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)

# 4. Turn the text into numbers!
# We FIT (learn the vocabulary) and TRANSFORM the training data
X_train = vectorizer.fit_transform(X_train_text)

# We ONLY TRANSFORM the test data (we don't let it learn new vocabulary from the test set)
X_test = vectorizer.transform(X_test_text)

print(f"Training data shape: {X_train.shape}")
print(f"Testing data shape: {X_test.shape}")

Data successfully split!
Training data shape: (66758, 5000)
Testing data shape: (16690, 5000)


In [4]:
from sklearn.naive_bayes import MultinomialNB

# 1. Initialize the Naive Bayes model
bayes_model = MultinomialNB()

# 2. Train the model!
print("Training the Naive Bayes model...")
bayes_model.fit(X_train, y_train)

print("Training complete! Bayes' Theorem has been applied.")


Training the Naive Bayes model...
Training complete! Bayes' Theorem has been applied.


In [ ]:
# 1. Instead of getting a flat 0 or 1, we ask for the raw probability percentages
# predict_proba returns two columns: [Probability of Ham, Probability of Spam]
probabilities = bayes_model.predict_proba(X_test)

# 2. Extract just the "Probability of Spam" column NOT the "HAM" probability
spam_probabilities = probabilities[:, 1]

# 3. Set our strict custom threshold (e.g., 90% confidence)
custom_threshold = 0.79

# 4. Create new predictions based on this strict rule
# If the probability is greater than 0.79, mark as True (Spam/1), else False (Ham/0)
strict_predictions = (spam_probabilities >= custom_threshold).astype(int)

# 5. Let's see how our Confusion Matrix changed!
print("New Confusion Matrix (90% Threshold):")
print(confusion_matrix(y_test, strict_predictions))

New Confusion Matrix (90% Threshold):
[[7873   65]
 [ 988 7764]]


In [ ]:
import joblib

# 1. Let's test it on brand new, custom emails! 
new_emails = [
    "Hey, are we still on for the meeting tomorrow at 3? Let me know.",
    "URGENT: Your bank account has been compromised. Click here to claim your $1000 prize!",
    "Can you please review the attached invoice by Friday?"
]

# 2. PRE-PROCESSING: We MUST translate this new English into math using the exact same Vectorizer we trained earlier.
# Notice we use .transform() and NOT .fit_transform(). We don't want it learning new words, just applying what it knows.
new_emails_matrix = vectorizer.transform(new_emails)

# 3. Get the probabilities from your trained Bayes model
new_probabilities = bayes_model.predict_proba(new_emails_matrix)[:, 1]

# 4. Apply Custom 0.79 threshold to make the final decision
print("--- LIVE INFERENCE RESULTS ---\n")
for i, email in enumerate(new_emails):
    
    prob = new_probabilities[i]
    # If the probability is 0.79 or higher, label it SPAM. Otherwise, HAM.
    is_spam = "SPAM" if prob >= 0.79 else "HAM (Safe)"
    
    print(f"Text: '{email}'")
    print(f"Prediction: {is_spam} (Spam Probability: {prob*100:.1f}%)\n")

# 5. Save the "Brain" (Model) and the "Translator" (Vectorizer) to your computer
joblib.dump(bayes_model, 'spam_classifier_model.pkl')
joblib.dump(vectorizer, 'text_vectorizer.pkl')

print("--- SAVED ---")
print("Model and Vectorizer successfully saved to disk as .pkl files!")

--- LIVE INFERENCE RESULTS ---

Text: 'Hey, are we still on for the meeting tomorrow at 3? Let me know.'
Prediction: HAM (Safe) (Spam Probability: 22.8%)

Text: 'URGENT: Your bank account has been compromised. Click here to claim your $1000 prize!'
Prediction: SPAM (Spam Probability: 95.8%)

Text: 'Can you please review the attached invoice by Friday?'
Prediction: HAM (Safe) (Spam Probability: 5.9%)

--- SAVED ---
Model and Vectorizer successfully saved to disk as .pkl files!
